# Canopy DiL Download via Python

This notebook demonstrates the process of:
- Finding your tenant's current sim version
- Comparing the current sim version to the latest sim version
- Getting the list of available downloads for your tenant on the selected sim version (either your tenant or latest)
- Downloading DiL for your tenant on the specified sim version
- Extract the downloaded ZIP file and get the `CanopyDiL.lic` file out of the downloaded package.


## Authentication Setup
Create and authenticate the Canopy session used by the API examples below.

In [ ]:
import canopy
import logging
import nest_asyncio

logging.basicConfig(level=logging.INFO)
nest_asyncio.apply()
authentication_data = canopy.prompt_for_authentication()
session = canopy.Session(authentication_data)
session.authentication.authenticate()

## Find the Tenant's Current Sim Version
Read the authenticated tenant's configured simulation version through the Canopy session cache.

In [ ]:
tenant_id = session.authentication.tenant_id
current_sim_version = session.tenant_sim_version.get(tenant_id)

print(f"Tenant ID: {tenant_id}")
print(f"Current tenant sim version: {current_sim_version}")

## Compare Against the Latest Sim Version
Query the platform-current sim version with no tenant filter and compare it with the tenant-specific value.

In [ ]:
sim_version_api = canopy.openapi.SimVersionApi(session.sync_client)
latest_sim_version = sim_version_api.sim_version_get_sim_version()

print(f"Tenant sim version: {current_sim_version}")
print(f"Latest platform sim version: {latest_sim_version}")
if current_sim_version == latest_sim_version:
    print("The tenant is already on the latest sim version.")
else:
    print("A newer platform sim version is available.")

## List Available Downloads
Choose the tenant's current or platform-latest version, then list that version's tenant-authorized downloads.

In [ ]:
DOWNLOAD_VERSION = "tenant"  # Set to "latest" to use the platform-current version.
if DOWNLOAD_VERSION == "tenant":
    selected_sim_version = current_sim_version
elif DOWNLOAD_VERSION == "latest":
    selected_sim_version = latest_sim_version
else:
    raise ValueError("DOWNLOAD_VERSION must be either 'tenant' or 'latest'.")

downloads_result = sim_version_api.sim_version_get_downloads(
    sim_version=selected_sim_version,
    tenant_id=tenant_id,
)
available_downloads = downloads_result.downloads or []

print(f"Available downloads for sim version {selected_sim_version}:")
for item in available_downloads:
    print(f"- {item.name}")

## Download the DiL ZIP
Select the exact package for the encryption setting, enter a local destination folder, and download the matching signed URL.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

ENCRYPTION_ENABLED = False
if ENCRYPTION_ENABLED:
    expected_download_name = (
        f"Canopy-Standard-DiL-EncryptionEnabled-Canopy-{selected_sim_version}.zip"
    )
else:
    expected_download_name = f"Canopy-Standard-DiL-{selected_sim_version}.zip"

dil_download = next(
    (item for item in available_downloads if item.name == expected_download_name),
    None,
)
if dil_download is None:
    raise FileNotFoundError(
        f"The selected DiL package is not available: {expected_download_name}"
    )

download_directory_input = input("Enter the local folder for the DiL ZIP: ").strip()
if not download_directory_input:
    raise ValueError("A local download folder is required.")
download_directory = Path(download_directory_input).expanduser()
download_directory.mkdir(parents=True, exist_ok=True)
download_path = download_directory / expected_download_name

urlretrieve(dil_download.url, download_path)
print(f"Downloaded {download_path} ({download_path.stat().st_size:,} bytes).")

## Extract the DiL License
Copy the ZIP's root `CanopyDiL.lic` entry into the same folder as the downloaded ZIP.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

license_path = download_path.with_name("CanopyDiL.lic")
with ZipFile(download_path) as dil_zip:
    license_path.write_bytes(dil_zip.read("CanopyDiL.lic"))

print(f"Extracted license: {license_path.resolve()}")